# eWaterCycle containerized model

This notebook is to showcase/test the containerized model.

We import the SWMM model:

In [ ]:
from ewatercycle.models import SWMM

To be able to initialize the model, we need an input **file.inp**.
This file is created with QGIS, see [documentation](https://github.com/Jannik-Schilling/generate_swmm_inp/blob/main/README.md) for more info on that.

eWaterCycle has wrapped the [pySWMM](https://www.pyswmm.org/) Simulation object into BMI. You should be able to just use it like a Simulation object. The following notebook showcases the Latte example for the pySWMM website.

In [ ]:
import ewatercycle
from ewatercycle.base.parameter_set import ParameterSet

from pathlib import Path
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

swmm_dir = Path.home() / "ewatercycle-swmm"
swmm_dir.mkdir(parents=True, exist_ok=True)

input_file = swmm_dir / "example.inp"

parameters = ParameterSet(
    name="SWMM_parameter_files",
    directory=swmm_dir,
    config=Path(input_file),
    target_model='swmm',
)

Now we can start the model. The `SWMM` class uses a container hosted on the Github container registry.



In [ ]:
model = SWMM(parameter_set=parameters)
cfg_file, _ = model.setup()

In [ ]:
model.initialize(cfg_file)

In [ ]:
model.get_component_name()

To test the model, we'll make a hydrograph:

In [ ]:
n_sc    = model.get_grid_size(0)
n_nodes = model.get_grid_size(1)
n_links = model.get_grid_size(2)

times       = []
sc_runoff   = []
node_depth  = []
node_flood  = []
link_flow   = []

while model.time < model.end_time:
    times.append(model.get_current_time())
    sc_runoff.append(model.get_value("subcatchment_runoff"))
    node_depth.append(model.get_value("node_depth"))
    node_flood.append(model.get_value("node_flooding"))
    link_flow.append( model.get_value("link_flow"))
    model.update()

# Convert to arrays and a pandas time axis
hr         = pd.to_datetime([datetime.datetime.fromtimestamp(t, tz=datetime.timezone.utc) for t in times])
sc_runoff  = np.array(sc_runoff)   # shape: (n_steps, n_sc)
node_depth = np.array(node_depth)  # shape: (n_steps, n_nodes)
node_flood = np.array(node_flood)
link_flow  = np.array(link_flow)   # shape: (n_steps, n_links)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("SWMM BMI Wrapper — Latte Example Simulation (6-hour triangular storm)", fontsize=13)

sc_labels   = [f"S{i+1}" for i in range(n_sc)]
node_labels = [f"Node {i+1}" for i in range(n_nodes)]
link_labels = [f"C{i+1}" for i in range(n_links)]

# Subcatchment runoff  [m3/s → L/s for readability]
ax = axes[0, 0]
for i, lbl in enumerate(sc_labels):
    ax.plot(hr, sc_runoff[:, i] * 1000, label=lbl)
ax.set_ylabel("Runoff [L/s]")
ax.set_title("Subcatchment Runoff")
ax.legend()

# Node water depth
ax = axes[0, 1]
for i, lbl in enumerate(node_labels):
    ax.plot(hr, node_depth[:, i], label=lbl)
ax.set_ylabel("Depth [m]")
ax.set_title("Node Water Depth")
ax.legend()

# Node flooding
ax = axes[1, 0]
for i, lbl in enumerate(node_labels):
    ax.plot(hr, node_flood[:, i] * 1000, label=lbl)
ax.set_ylabel("Flooding [L/s]")
ax.set_title("Node Flooding")
ax.legend()

# Link flow
ax = axes[1, 1]
for i, lbl in enumerate(link_labels):
    ax.plot(hr, link_flow[:, i] * 1000, label=lbl)
ax.set_ylabel("Flow [L/s]")
ax.set_title("Conduit Flow")
ax.legend()

for ax in axes.flat:
    ax.set_xlabel("Time")
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
# plt.savefig("swmm_bmi_results.png", dpi=100, bbox_inches="tight")
plt.show()

To make sure that the container running the model is nicely shut down and doesn't keep running in the background (taking up resources) we need to 'finalize' it.

In [ ]:
model.finalize()

## Inflow

Let us check the inflow by setting the value.

In [ ]:
EXTERNAL_INFLOW_M3S = 0.05  # 50 L/s injected at every node (on top of storm runoff)

model2 = SWMM(parameter_set=parameters)
cfg_file, _ = model2.setup()

model2.initialize(cfg_file)

In [ ]:
n_nodes2 = model2.get_grid_size(1)
n_links2 = model2.get_grid_size(2)
times2      = []
node_depth2 = []
node_flood2 = []
link_flow2  = []

while model2.get_current_time() < model2.get_end_time():
    model2.set_value(
        "node_lateral_inflow",
        np.full(n_nodes2, EXTERNAL_INFLOW_M3S)
    )
    model2.update()
    times2.append(model2.get_current_time())
    node_depth2.append(model2.get_value("node_depth"))
    node_flood2.append(model2.get_value("node_flooding"))
    link_flow2.append( model2.get_value("link_flow"))

node_depth2 = np.array(node_depth2)
node_flood2 = np.array(node_flood2)
link_flow2  = np.array(link_flow2)
hr2 = pd.to_datetime([datetime.datetime.fromtimestamp(t, tz=datetime.timezone.utc) for t in times2])

print("External inflow run complete.")
print(f"Peak outfall flow (built-in) : {link_flow[:, -1].max()*1000:.1f} L/s")
print(f"Peak outfall flow (+ inflow) : {link_flow2[:, -1].max()*1000:.1f} L/s")

In [ ]:
model2.finalize()

In [ ]:
# If you wish to remove the temporary files the model creates:
# !rm -rf swmm_20*